## 📦 1. Import Thư viện

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import json
import re
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

print(" Deep Learning Bi-LSTM (Ultimate Local Data Fusion)...")

 Deep Learning Bi-LSTM (Ultimate Local Data Fusion)...


## 📂 2. dữ liệu từ Ổ D:

In [3]:
print("📂 Đang tiến hành hút và dung hợp dữ liệu...")
dfs = []

# --- A. Xử lý nhóm file HttpParamsDataset ---
def normalize_http_label(label):
    label = str(label).lower().strip()
    if label == 'norm': return 'Normal'
    if 'xss' in label or 'js-syntax' in label: return 'XSS'  # Gộp JS-Syntax vào XSS
    if 'sql' in label: return 'SQLi'
    if 'cmd' in label or 'exec' in label: return 'Command Injection'
    if 'path' in label or 'traversal' in label: return 'Path Traversal'
    if 'ssrf' in label: return 'SSRF'
    return label.upper()

http_files = [
    r"D:\AI\clawweb\data\httpparagram\payload_train.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_test.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_test_lexical.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_full.csv"
]

for f in http_files:
    if os.path.exists(f):
        df_temp = pd.read_csv(f)
        df_temp = df_temp[['payload', 'attack_type']].rename(columns={'payload': 'text', 'attack_type': 'label'})
        df_temp['label'] = df_temp['label'].apply(normalize_http_label)
        dfs.append(df_temp)
        print(f"✅ Đã nạp {len(df_temp)} dòng từ {os.path.basename(f)}")
    else:
        print(f"⚠️ Không tìm thấy: {f}")

# --- B. Xử lý bộ XSS Kaggle ---
path_xss = r"D:\AI\clawweb\data\xss\XSS_dataset.csv"
if os.path.exists(path_xss):
    df_xss = pd.read_csv(path_xss)
    df_xss = df_xss[['Sentence', 'Label']].rename(columns={'Sentence': 'text', 'Label': 'label'})
    df_xss['label'] = df_xss['label'].map({1: 'XSS', 0: 'Normal'})
    dfs.append(df_xss)
    print(f"✅ Đã nạp {len(df_xss)} dòng từ XSS_dataset.csv")

# --- C. Xử lý bộ Command Injection Kaggle ---
path_cmd = r"D:\AI\clawweb\data\oscommand\command injection.csv"
if os.path.exists(path_cmd):
    df_cmd = pd.read_csv(path_cmd)
    df_cmd = df_cmd[['sentence', 'Label']].rename(columns={'sentence': 'text', 'Label': 'label'})
    df_cmd['label'] = df_cmd['label'].map({1: 'Command Injection', 0: 'Normal'})
    dfs.append(df_cmd)
    print(f"✅ Đã nạp {len(df_cmd)} dòng từ command injection.csv")

# --- D. Xử lý bộ SQL Injection (Dự đoán đường dẫn) ---
path_sqli = r"D:\AI\clawweb\data\sqli\Modified_SQL_Dataset.csv"
if not os.path.exists(path_sqli):
    path_sqli = r"D:\AI\clawweb\data\sql_injection\Modified_SQL_Dataset.csv"

if os.path.exists(path_sqli):
    df_sqli = pd.read_csv(path_sqli)
    df_sqli = df_sqli.rename(columns={"Query": "text", "Label": "label"})[["text", "label"]]
    df_sqli["label"] = df_sqli["label"].map({1: "SQLi", 0: "Normal", "1": "SQLi", "0": "Normal"})
    dfs.append(df_sqli)
    print(f"✅ Đã nạp {len(df_sqli)} dòng từ Modified_SQL_Dataset.csv")

# --- E. Xử lý JSONL (Dùng Regex Bất Tử chống lỗi file Kaggle) ---
path_jsonl = r"D:\AI\clawweb\data\web_payloads\WEB_APPLICATION_PAYLOADS.jsonl"
if os.path.exists(path_jsonl):
    json_payloads = []
    with open(path_jsonl, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Dùng Biểu thức chính quy (Regex) để cào dữ liệu thay vì json.loads
    # Việc này giúp bỏ qua hoàn toàn các dòng bị lỗi thiếu dấu phẩy của Kaggle
    matches = re.finditer(r'"payload"\s*:\s*"(.*?)"[\s\S]*?"type"\s*:\s*"(.*?)"', content)
    for match in matches:
        payload_text = match.group(1).replace('\\"', '"')
        payload_type = match.group(2).upper()
        if 'SSRF' in payload_type:
            json_payloads.append((payload_text, 'SSRF'))
        elif 'CSRF' in payload_type:
            json_payloads.append((payload_text, 'CSRF'))
    if json_payloads:
        df_json = pd.DataFrame(json_payloads, columns=["text", "label"])
        df_json = pd.concat([df_json] * 50, ignore_index=True) # Nhân bản lên 50 lần
        dfs.append(df_json)
        print(f"✅ Đã cào & nhân bản {len(df_json)} dòng SSRF/CSRF bằng Regex Engine")
    else:
        print("⚠️ Không trích xuất được SSRF/CSRF nào từ Regex.")

# --- GOM TẤT CẢ VÀ DỌN DẸP ---
if not dfs:
    raise ValueError("❌ LỖI NGHIÊM TRỌNG: Không đọc được bất kỳ file nào từ ổ D:!")

df = pd.concat(dfs, ignore_index=True)
df = df.drop_duplicates().dropna() # Xoa sach rac trung lap
df['text'] = df['text'].astype(str)

initial_len = len(df)

print(f"\n🧹 Đã dọn dẹp {initial_len - len(df)} dòng copy/paste trùng lặp.")
print(f"📊 Tổng lực lượng Data hiện có: {len(df)} dòng.")
print("\n🔥 Phân bổ Các Nhãn Lỗ Hổng (Đã Sạch Sẽ):")
print(df['label'].value_counts())

# Bơm thêm URL sạch vào lớp Normal để AI không nhận nhầm URL thành XSS/SSRF
normal_urls = pd.DataFrame({
    'text': [
        'https://www.google.com/search?q=cat',
        'http://localhost:5173/dashboard/users?id=123',
        'https://uet.vnu.edu.vn/category/tin-tuc/',
        'http://portal.edu.vn/api/docs/file.pdf',
        'https://github.com/search?q=machine+learning'
    ] * 10, # CHỈ NHÂN BẢN 10 LẦN
    'label': ['Normal'] * 50
})

df = pd.concat([df, normal_urls], ignore_index=True)


📂 Đang tiến hành hút và dung hợp dữ liệu...
✅ Đã nạp 20712 dòng từ payload_train.csv
✅ Đã nạp 10355 dòng từ payload_test.csv
✅ Đã nạp 1106 dòng từ payload_test_lexical.csv
✅ Đã nạp 31067 dòng từ payload_full.csv
✅ Đã nạp 13686 dòng từ XSS_dataset.csv
✅ Đã nạp 2106 dòng từ command injection.csv
✅ Đã nạp 30919 dòng từ Modified_SQL_Dataset.csv
✅ Đã cào & nhân bản 9950 dòng SSRF/CSRF bằng Regex Engine

🧹 Đã dọn dẹp 0 dòng copy/paste trùng lặp.
📊 Tổng lực lượng Data hiện có: 67148 dòng.

🔥 Phân bổ Các Nhãn Lỗ Hổng (Đã Sạch Sẽ):
label
Normal               36359
SQLi                 21929
XSS                   7873
Command Injection      520
Path Traversal         290
CSRF                    92
SSRF                    85
Name: count, dtype: int64


In [21]:
import pandas as pd
df = pd.read_csv("D:\AI\clawweb\data\oscommand\command injection.csv")
print(df.columns.tolist())
print(df.head(3))

['sentence', 'Label']
                                            sentence  Label
0  &lt;!--#exec%20cmd=&quot;/bin/cat%20/etc/passw...      1
1  &lt;!--#exec%20cmd=&quot;/bin/cat%20/etc/shado...      1
2        &lt;!--#exec%20cmd=&quot;/usr/bin/id;--&gt;      1


## ⚖️ 3. Xử lý Lệch Dữ Liệu (Undersampling)

In [4]:
print("📂 Đang tiến hành hút và dung hợp dữ liệu...")
import html as html_lib
from urllib.parse import unquote
dfs = []

# ── HELPER: decode HTML entities + URL encode ────────────
def clean_payload(text):
    if not isinstance(text, str):
        return str(text)
    text = html_lib.unescape(text)   # &lt; → <
    text = unquote(text)              # %20 → space
    text = unquote(text)              # double encoded: %2520 → %20 → space
    return text.strip()

# --- A. Xử lý nhóm file HttpParamsDataset ---
def normalize_http_label(label):
    label = str(label).lower().strip()
    if label == 'norm': return 'Normal'
    if 'xss' in label or 'js-syntax' in label: return 'XSS'
    if 'sql' in label: return 'SQLi'
    if 'cmd' in label or 'exec' in label: return 'Command Injection'
    if 'path' in label or 'traversal' in label: return 'Path Traversal'
    if 'ssrf' in label: return 'SSRF'
    if 'csrf' in label: return 'CSRF'
    return None  # None thay vì label.upper() → sẽ drop sau

http_files = [
    r"D:\AI\clawweb\data\httpparagram\payload_train.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_test.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_test_lexical.csv",
    r"D:\AI\clawweb\data\httpparagram\payload_full.csv"
]
for f in http_files:
    if os.path.exists(f):
        df_temp = pd.read_csv(f)
        df_temp = df_temp[['payload', 'attack_type']].rename(
            columns={'payload': 'text', 'attack_type': 'label'})
        df_temp['text']  = df_temp['text'].apply(clean_payload)
        df_temp['label'] = df_temp['label'].apply(normalize_http_label)
        df_temp = df_temp[df_temp['label'].notna()]
        dfs.append(df_temp)
        print(f"✅ Đã nạp {len(df_temp)} dòng từ {os.path.basename(f)}")
    else:
        print(f"⚠️ Không tìm thấy: {f}")

# --- B. XSS Kaggle ---
path_xss = r"D:\AI\clawweb\data\xss\XSS_dataset.csv"
if os.path.exists(path_xss):
    df_xss = pd.read_csv(path_xss)
    df_xss = df_xss[['Sentence', 'Label']].rename(
        columns={'Sentence': 'text', 'Label': 'label'})
    df_xss['text']  = df_xss['text'].apply(clean_payload)
    df_xss['label'] = df_xss['label'].map({1: 'XSS', 0: 'Normal'})
    dfs.append(df_xss)
    print(f"✅ Đã nạp {len(df_xss)} dòng từ XSS_dataset.csv")

# --- C. Command Injection Kaggle — decode TRƯỚC dedup ---
path_cmd = r"D:\AI\clawweb\data\oscommand\command injection.csv"
if os.path.exists(path_cmd):
    df_cmd = pd.read_csv(path_cmd)
    df_cmd = df_cmd[['sentence', 'Label']].rename(
        columns={'sentence': 'text', 'Label': 'label'})
    df_cmd['text']  = df_cmd['text'].apply(clean_payload)  # FIX: decode trước
    df_cmd['label'] = df_cmd['label'].map({1: 'Command Injection', 0: 'Normal'})
    dfs.append(df_cmd)
    print(f"✅ Đã nạp {len(df_cmd)} dòng từ command injection.csv")

# --- D. SQL Injection ---
path_sqli = r"D:\AI\clawweb\data\sqli\Modified_SQL_Dataset.csv"
if not os.path.exists(path_sqli):
    path_sqli = r"D:\AI\clawweb\data\sql_injection\Modified_SQL_Dataset.csv"
if os.path.exists(path_sqli):
    df_sqli = pd.read_csv(path_sqli)
    df_sqli = df_sqli.rename(columns={"Query": "text", "Label": "label"})[["text", "label"]]
    df_sqli['text']  = df_sqli['text'].apply(clean_payload)
    df_sqli["label"] = df_sqli["label"].map(
        {1: "SQLi", 0: "Normal", "1": "SQLi", "0": "Normal"})
    dfs.append(df_sqli)
    print(f"✅ Đã nạp {len(df_sqli)} dòng từ Modified_SQL_Dataset.csv")

# --- E. SSRF/CSRF từ JSONL (giữ nguyên logic gốc) ---
path_jsonl = r"D:\AI\clawweb\data\web_payloads\WEB_APPLICATION_PAYLOADS.jsonl"
if os.path.exists(path_jsonl):
    json_payloads = []
    with open(path_jsonl, 'r', encoding='utf-8') as f:
        content = f.read()
    matches = re.finditer(
        r'"payload"\s*:\s*"(.*?)"[\s\S]*?"type"\s*:\s*"(.*?)"', content)
    for match in matches:
        payload_text = match.group(1).replace('\\"', '"')
        payload_type = match.group(2).upper()
        if 'SSRF' in payload_type:
            json_payloads.append((payload_text, 'SSRF'))
        elif 'CSRF' in payload_type:
            json_payloads.append((payload_text, 'CSRF'))
    if json_payloads:
        df_json = pd.DataFrame(json_payloads, columns=["text", "label"])
        # KHÔNG nhân bản 50 lần nữa — dùng data thật + thêm từ GitHub thay thế

        # LỌC CSRF: loại bỏ mẫu chứa XSS event handlers
        xss_pattern = re.compile(r'(?i)(onerror|onload|onclick|onmouseover|onfocus|<script>|alert\(|document\.cookie)')
        before_filter = len(df_json[df_json['label'] == 'CSRF'])
        df_json = df_json[~((df_json['label'] == 'CSRF') & df_json['text'].apply(lambda x: bool(xss_pattern.search(str(x)))))]
        after_filter = len(df_json[df_json['label'] == 'CSRF'])
        print(f"  🧹 Lọc CSRF: {before_filter} → {after_filter} (loại {before_filter - after_filter} mẫu XSS lẫn)")
        dfs.append(df_json)
        print(f"✅ Đã cào {len(df_json)} dòng SSRF/CSRF từ JSONL")
    else:
        print("⚠️ Không trích xuất được SSRF/CSRF nào từ JSONL.")

# --- F. DATA MỚI TỪ GITHUB ---
# Download các file này về D:\AI\clawweb\data\new\ trước khi chạy:
#
# CMDi:  https://raw.githubusercontent.com/omurugur/OS_Command_Payload_List/master/OS-Command-Fuzzing.txt
# Path:  https://raw.githubusercontent.com/swisskyrepo/PayloadsAllTheThings/master/Path%20Traversal/Intruder/path-traversal.txt
# SSRF:  https://raw.githubusercontent.com/swisskyrepo/PayloadsAllTheThings/master/Server%20Side%20Request%20Forgery/Intruder/ssrf.txt

def load_txt_as_df(filepath, label):
    if not os.path.exists(filepath):
        print(f"  ⏭️  Chưa có: {os.path.basename(filepath)} — bỏ qua")
        return None
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    df_new = pd.DataFrame({
        'text': [clean_payload(l) for l in lines],
        'label': label
    })
    df_new = df_new[df_new['text'].str.len() >= 3]
    df_new = df_new.drop_duplicates(subset='text')
    print(f"  ✅ {len(df_new)} payload mới từ {os.path.basename(filepath)} → [{label}]")
    return df_new

print("\n📦 Nạp data mới từ GitHub...")
new_data_dir = r"D:\AI\clawweb\data\new"
for fname, label in [
    ("OS-Command-Fuzzing.txt",  "Command Injection"),
    ("path-traversal.txt",      "Path Traversal"),
    ("ssrf.txt",                "SSRF"),
    ("SSRF.txt",                "SSRF"),
    ("traversal.txt",           "Path Traversal"),
    # ── MỚI: MODERN ATTACKS ──
    ("ssti.txt",                "SSTI"),
    ("nosqli.txt",              "NoSQLi"),
    ("xxe.txt",                 "XXE"),
    ("jwt.txt",                 "JWTAuth"),
]:
    result = load_txt_as_df(os.path.join(new_data_dir, fname), label)
    if result is not None:
        dfs.append(result)

# --- F2. DATA MỚI TỪ DATACOLLECT ---
path_new_variants = r"d:\AI\ai_security\data_new_variants.csv"
if os.path.exists(path_new_variants):
    df_new_variants = pd.read_csv(path_new_variants)
    # Lọc các dòng hợp lệ
    df_new_variants = df_new_variants.dropna(subset=['payload', 'label'])
    df_new_variants['text'] = df_new_variants['payload'].apply(clean_payload)
    df_new_variants = df_new_variants[df_new_variants['text'].str.len() >= 3]
    df_new_variants = df_new_variants[['text', 'label']]
    dfs.append(df_new_variants)
    print(f"✅ Đã nạp {len(df_new_variants)} dòng từ data_new_variants.csv")
else:
    print(f"⚠️ Chưa có file: {path_new_variants}")

# --- NORMAL URLs bổ sung (giữ nguyên logic gốc) ---
# Normal URLs — giúp model phân biệt URL bình thường vs SSRF
# Normal URLs — MASSIVE list để model hiểu 'URL != tấn công'
normal_url_samples = [
    'https://www.google.com/search?q=cat',
    'https://www.google.com/search?q=machine+learning',
    'https://www.google.com/search?q=python+tutorial',
    'https://www.google.com/maps?q=hanoi',
    'https://www.youtube.com/watch?v=dQw4w9WgXcQ',
    'https://www.youtube.com/results?search_query=flask',
    'https://www.wikipedia.org/wiki/Python',
    'https://en.wikipedia.org/wiki/Artificial_intelligence',
    'https://stackoverflow.com/questions/12345',
    'https://stackoverflow.com/search?q=pandas+dataframe',
    'https://github.com/tensorflow/tensorflow',
    'https://github.com/search?q=machine+learning',
    'https://mail.google.com/mail/u/0/',
    'https://drive.google.com/file/d/abc123/view',
    'https://docs.google.com/document/d/xyz/edit',
    'http://localhost:5173/dashboard/users?id=123',
    'http://localhost:8080/api/v1/products',
    'http://localhost:3000/health',
    'http://localhost:8000/docs',
    'http://localhost:5000/login',
    'https://uet.vnu.edu.vn/category/tin-tuc/',
    'https://hus.vnu.edu.vn/nghien-cuu/bai-bao',
    'http://portal.edu.vn/api/docs/file.pdf',
    'https://example.com/login?redirect=home',
    'https://example.com/api/users/123',
    'https://example.com/products?page=2&sort=name',
    'http://192.168.1.1/admin/settings',
    'http://192.168.0.100/printer/status',
    'http://10.0.0.1:8080/api/v1/users',
    'http://company.internal:9090/metrics',
    'http://myapp.local:3000/health',
    'https://cdn.jsdelivr.net/npm/vue@3',
    'https://fonts.googleapis.com/css?family=Roboto',
    'http://api.weather.com/v1/forecast?city=hanoi',
    'https://jsonplaceholder.typicode.com/posts/1',
    'https://api.github.com/users/octocat',
    'https://httpbin.org/get?foo=bar',
    'http://worldtimeapi.org/api/timezone/Asia/Ho_Chi_Minh',
    'https://pokeapi.co/api/v2/pokemon/pikachu',
    'https://dog.ceo/api/breeds/image/random',
    'https://www.facebook.com/profile.php?id=100',
    'https://twitter.com/search?q=python',
    'https://www.linkedin.com/in/username',
    'https://www.reddit.com/r/learnpython',
    'https://medium.com/@user/article-title',
    'https://dev.to/search?q=flask+tutorial',
    'https://www.npmjs.com/package/express',
    'https://pypi.org/project/tensorflow/',
    'https://hub.docker.com/_/python',
    'https://www.amazon.com/dp/B09V3KXJPB',
]
normal_urls = pd.DataFrame({
    'text': normal_url_samples * 40,  # 50 URL * 40 = 2000 mẫu Normal URL
    'label': ['Normal'] * (len(normal_url_samples) * 40)
})
dfs.append(normal_urls)
print(f"  ✅ Thêm {len(normal_urls)} mẫu Normal URL (chống SSRF/CSRF false positive)")


# ── MẪU SQLi NGẮN (chống false negative cho payload đơn giản) ──
short_sqli_patterns = [
    # Pattern: ' OR / ' AND (dạng cơ bản nhất)
    "admin' OR 1", "admin' OR '1'='1", "' OR 1=1--", "' OR 1=1#",
    "' OR 'a'='a", "' OR ''='", "1' OR '1'='1", "' OR true--",
    "admin'--", "' OR 1--", "admin' OR 1=1", "' OR 1#",
    # Pattern: UNION (dạng ngắn)
    "' UNION SELECT 1--", "' UNION SELECT null--", "1' UNION SELECT 1,2--",
    # Pattern: comment / terminate
    "admin'--", "admin'#", "';--", "1';--",
    # Pattern: tautology ngắn
    "' OR 1=1", "' OR '1'='1'", "' OR 'x'='x", "1 OR 1=1",
    "' OR 'a'='a'--", "\" OR 1=1--", "\" OR \"a\"=\"a",
    # Biến thể với số
    "1' OR 1", "2' OR 1=1", "999' OR 1=1--", "0' OR 1=1#",
    # Biến thể admin
    "admin' or '1'='1", "admin' or 1=1--", "admin'or 1=1#",
    "user' OR 1=1--", "test' OR 1=1--", "root' OR 1=1--",
    # Dạng AND (ít phổ biến hơn nhưng vẫn là SQLi)
    "' AND 1=1--", "' AND '1'='1", "admin' AND 1=1--",
]
# Nhân bản x20 để có ~800 mẫu, đủ trọng lượng trong dataset
short_sqli_df = pd.DataFrame({
    'text': [clean_payload(p) for p in short_sqli_patterns] * 20,
    'label': ['SQLi'] * (len(short_sqli_patterns) * 20)
})
dfs.append(short_sqli_df)
print(f"  ✅ Thêm {len(short_sqli_df)} mẫu SQLi ngắn (short-form)")

# ── MẪU XSS NGẮN (chống nhầm sang CSRF) ──
short_xss_patterns = [
    "<img src=x onerror=alert(1)>", "<img src=x onerror='alert(1)'>",
    "<img/src=x onerror=alert(1)>", "<IMG SRC=x ONERROR=alert(1)>",
    "<svg onload=alert(1)>", "<svg/onload=alert(1)>",
    "<body onload=alert(1)>", "<input onfocus=alert(1) autofocus>",
    "<details open ontoggle=alert(1)>", "<marquee onstart=alert(1)>",
    "<video src=x onerror=alert(1)>", "<audio src=x onerror=alert(1)>",
]
short_xss_df = pd.DataFrame({
    'text': [clean_payload(p) for p in short_xss_patterns] * 20,
    'label': ['XSS'] * (len(short_xss_patterns) * 20)
})
dfs.append(short_xss_df)

# ── CSRF: chỉ giữ dạng AUTO-SUBMIT FORM (đặc trưng riêng, không overlap URL/XSS) ──
csrf_form_patterns = [
    "<form action='https://bank.com/transfer' method='POST' id='f'><input type='hidden' name='to' value='hacker'/><input type='hidden' name='amount' value='9999'/></form><script>document.getElementById('f').submit()</script>",
    "<form action='/api/change-password' method='POST'><input type='hidden' name='new_pass' value='hacked'/></form><script>document.forms[0].submit()</script>",
    "<form action='/admin/delete-user' method='POST'><input type='hidden' name='user_id' value='1'/></form><script>document.forms[0].submit()</script>",
    "<form action='https://target.com/settings/email' method='POST'><input name='email' value='evil@hacker.com'/></form><script>document.forms[0].submit()</script>",
    "<form method='POST' action='/transfer'><input type='hidden' name='to' value='attacker'/></form><script>document.forms[0].submit()</script>",
    "<form action='/api/grant-admin' method='POST'><input type='hidden' name='role' value='admin'/></form><script>document.forms[0].submit()</script>",
    "<iframe src='https://target.com/transfer?to=hacker&amount=100' style='display:none'></iframe>",
    "<iframe src='/api/delete-account?confirm=1' width='0' height='0'></iframe>",
]
csrf_form_df = pd.DataFrame({
    'text': [clean_payload(p) for p in csrf_form_patterns] * 15,
    'label': ['CSRF'] * (len(csrf_form_patterns) * 15)
})
dfs.append(csrf_form_df)
print(f"  ✅ Thêm {len(csrf_form_df)} mẫu CSRF auto-submit form")
print(f"  ✅ Thêm {len(short_xss_df)} mẫu XSS event-handler (chống nhầm CSRF)")

# ══════════════════════════════════════════════════════════
# GOM TẤT CẢ + DỌN DẸP
# ══════════════════════════════════════════════════════════
if not dfs:
    raise ValueError("❌ Không đọc được bất kỳ file nào!")

df = pd.concat(dfs, ignore_index=True)
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() >= 3]

before_dedup = len(df)
df = df.drop_duplicates(subset='text')
print(f"\n🧹 Đã dọn dẹp {before_dedup - len(df)} dòng trùng lặp.")
print(f"📊 Tổng lực lượng Data hiện có: {len(df)} dòng.")
print("\n🔥 Phân bổ Nhãn (Trước cân bằng):")
print(df['label'].value_counts())

# ══════════════════════════════════════════════════════════
# CÂN BẰNG
# ══════════════════════════════════════════════════════════
print("\n⚖️ Đang thực hiện cân bằng dữ liệu nâng cao...")

target_count = 8000   # Tăng lên 8000 để giảm tỷ lệ mẫu lặp cho nhãn nhỏ
new_dfs = []

for label in df['label'].unique():
    temp_df = df[df['label'] == label].copy()
    current_count = len(temp_df)

    if label == 'Normal':
        temp_df = temp_df.sample(n=min(current_count, 15000), random_state=42)

    elif current_count < target_count:
        multiplier = min(int(target_count / current_count) + 1, 3)  # Giới hạn x3
        temp_df = pd.concat([temp_df] * multiplier,
                             ignore_index=True).iloc[:target_count].copy()
        # Noise nhẹ để tránh exact duplicate sau khi nhân bản
        temp_df['text'] = temp_df['text'].apply(
            lambda x: x + ' ' * np.random.randint(0, 2))

    elif current_count > 16000:  # Downsample nhẹ: giữ tối đa 16,000
        # Downsample lớp quá lớn (SQLi ~21k) để không lấn át
        temp_df = temp_df.sample(n=16000, random_state=42)

    new_dfs.append(temp_df)

df = pd.concat(new_dfs, ignore_index=True).sample(
    frac=1, random_state=42).reset_index(drop=True)


# ── BƠM TRỰC TIẾP DATA QUAN TRỌNG VÀO CUỐI CÙNG (CHỐNG DROP_DUPLICATES) ──
print("\n💉 Đang bơm trực tiếp các mẫu đặc chủng vào tập dữ liệu cuối...")

# 1. Normal URLs (Chống SSRF false positive)
normal_urls = pd.DataFrame({
'text': normal_url_samples * 40,
'label': ['Normal'] * (len(normal_url_samples) * 40)
})

# 1.5 Normal VN Texts (Chống false positive nhận diện tiếng Việt thành JWT/XSS)
normal_vn_samples = [
    "Xin chào, tôi muốn tìm tài liệu NCKH",
    "Báo cáo tiến độ đồ án tốt nghiệp",
    "Đây là câu văn bình thường không có hàm ý tấn công gì cả",
    "Sinh viên đăng ký môn học trực tuyến",
    "Kết quả học tập học kỳ 1 năm 2026",
    "Tôi xin phép vắng mặt ngày hôm nay",
    "Một hai ba bốn năm sáu bảy tám chín mười",
    "Cho tôi hỏi chức năng này dùng như thế nào",
    "Chào buổi sáng mọi người",
    "Cộng hòa xã hội chủ nghĩa Việt Nam",
]
normal_vn_df = pd.DataFrame({
'text': normal_vn_samples * 50,
'label': ['Normal'] * (len(normal_vn_samples) * 50)
})

# 2. Short SQLi (Chống admin' OR 1 false negative)
short_sqli_df = pd.DataFrame({
'text': [clean_payload(p) for p in short_sqli_patterns] * 20,
'label': ['SQLi'] * (len(short_sqli_patterns) * 20)
})

# 3. Short XSS (Chống nhầm CSRF)
short_xss_df = pd.DataFrame({
'text': [clean_payload(p) for p in short_xss_patterns] * 20,
'label': ['XSS'] * (len(short_xss_patterns) * 20)
})

# 4. Pure CSRF Form
csrf_form_df2 = pd.DataFrame({
'text': [clean_payload(p) for p in csrf_form_patterns] * 20,
'label': ['CSRF'] * (len(csrf_form_patterns) * 20)
})

# Gộp tất cả
df = pd.concat([df, normal_urls, normal_vn_df, short_sqli_df, short_xss_df, csrf_form_df2], ignore_index=True).sample(
frac=1, random_state=42).reset_index(drop=True)
print(f"  ✅ Đã bơm an toàn: {len(normal_urls)} Normal, {len(short_sqli_df)} SQLi, {len(short_xss_df)} XSS, {len(csrf_form_df2)} CSRF.\n")
print("📊 Phân bổ nhãn MỚI NHẤT sau cân bằng:")
print(df['label'].value_counts())
print(f"\n✅ Tổng: {len(df)} mẫu | {df['label'].nunique()} nhãn")

📂 Đang tiến hành hút và dung hợp dữ liệu...
✅ Đã nạp 20712 dòng từ payload_train.csv
✅ Đã nạp 10355 dòng từ payload_test.csv
✅ Đã nạp 1106 dòng từ payload_test_lexical.csv
✅ Đã nạp 31067 dòng từ payload_full.csv
✅ Đã nạp 13686 dòng từ XSS_dataset.csv
✅ Đã nạp 2106 dòng từ command injection.csv
✅ Đã nạp 30919 dòng từ Modified_SQL_Dataset.csv
  🧹 Lọc CSRF: 99 → 18 (loại 81 mẫu XSS lẫn)
✅ Đã cào 118 dòng SSRF/CSRF từ JSONL

📦 Nạp data mới từ GitHub...
  ✅ 5539 payload mới từ OS-Command-Fuzzing.txt → [Command Injection]
  ✅ 103 payload mới từ path-traversal.txt → [Path Traversal]
  ✅ 224 payload mới từ ssrf.txt → [SSRF]
  ✅ 224 payload mới từ SSRF.txt → [SSRF]
  ✅ 103 payload mới từ traversal.txt → [Path Traversal]
  ⏭️  Chưa có: ssti.txt — bỏ qua
  ⏭️  Chưa có: nosqli.txt — bỏ qua
  ⏭️  Chưa có: xxe.txt — bỏ qua
  ⏭️  Chưa có: jwt.txt — bỏ qua
✅ Đã nạp 9706 dòng từ data_new_variants.csv
  ✅ Thêm 2000 mẫu Normal URL (chống SSRF/CSRF false positive)
  ✅ Thêm 780 mẫu SQLi ngắn (short-form)
 

## 🧠 4. Tiền xử lý & Chia tập (Tokenization)

In [23]:
le = LabelEncoder()
y = le.fit_transform(df['label'])
num_classes = len(le.classes_)

print("🧠 Đang xây dựng từ điển nhúng ký tự (Tokenizer)...")
MAX_WORDS = 10000  
MAX_LEN = 150      

# ═══ FIX DATA LEAKAGE: Split TRƯỚC khi fit tokenizer ═══
# Chia 70% train / 15% val / 15% test (stratify giữ cân bằng nhãn)
X_text = df['text'].values

X_text_train, X_temp, y_train, y_temp = train_test_split(
    X_text, y, test_size=0.3, random_state=42, stratify=y)
X_text_val, X_text_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Fit tokenizer CHỈ trên train set → test/val hoàn toàn "mù" vocabulary
tokenizer = Tokenizer(num_words=MAX_WORDS, char_level=True, oov_token='<OOV>')
tokenizer.fit_on_texts(X_text_train)

# Transform tất cả
X_train = pad_sequences(tokenizer.texts_to_sequences(X_text_train), maxlen=MAX_LEN, padding='post', truncating='post')
X_val   = pad_sequences(tokenizer.texts_to_sequences(X_text_val),   maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(tokenizer.texts_to_sequences(X_text_test),  maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Tập train: {X_train.shape}, Tập val: {X_val.shape}, Tập test: {X_test.shape}")

🧠 Đang xây dựng từ điển nhúng ký tự (Tokenizer)...
Tập train: (45950, 150), Tập val: (9846, 150), Tập test: (9847, 150)


## 🏗️ 5. Xây dựng Mạng Nơ-ron (Bi-LSTM)

In [24]:
print("🏗️ Đang lắp ráp bộ não nơ-ron Bi-LSTM...")
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5), 
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

🏗️ Đang lắp ráp bộ não nơ-ron Bi-LSTM...


c:\Users\truon\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 🔥 6. Huấn luyện (Training)

In [25]:
from sklearn.utils import class_weight
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Tính trọng số gốc
raw_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

# ÁP DỤNG SMOOTHING (Làm trơn bằng Căn bậc 2)
smoothed_weights = np.sqrt(raw_weights)
class_weights_dict = dict(enumerate(smoothed_weights))

print("⚖️ Trọng số sau khi làm trơn:", class_weights_dict)

# ═══ FIX VALIDATION LEAK: Dùng val set riêng cho EarlyStopping ═══
print("🎬 Bắt đầu huấn luyện (Smoothed Class Weights + Proper Validation)...")
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val, y_val),  # Val set riêng — test set KHÔNG tham gia training
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

# Đánh giá trên TEST SET (hoàn toàn độc lập, không dùng để tune model)
print("\n📊 Đánh giá cuối cùng trên TEST SET (không dùng để tune):")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc:.4f}")


⚖️ Trọng số sau khi làm trơn: {0: np.float64(0.7944692056217904), 1: np.float64(4.494196501912461), 2: np.float64(0.7944692056217904), 3: np.float64(2.58002444718099), 4: np.float64(7.157255753566593), 5: np.float64(0.5371592235056327), 6: np.float64(2.069874944140898), 7: np.float64(1.4005341030929617), 8: np.float64(0.5485624592690803), 9: np.float64(1.7654871236039786), 10: np.float64(3.9116959622860903), 11: np.float64(0.7828137688990782), 12: np.float64(4.017436530523237)}
🎬 Bắt đầu huấn luyện (Smoothed Class Weights + Proper Validation)...
Epoch 1/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 93s 121ms/step - accuracy: 0.7703 - loss: 0.8371 - val_accuracy: 0.9014 - val_loss: 0.3236
Epoch 2/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 72s 101ms/step - accuracy: 0.9015 - loss: 0.4090 - val_accuracy: 0.9357 - val_loss: 0.2105
Epoch 3/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 74s 102ms/step - accuracy: 0.9225 - loss: 0.3097 - val_accuracy: 0.9402 - val_loss: 0.1845
Epoch 4/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 79s 110ms/step 

## 💾 7. Lưu Mô hình (Export vào repo)

In [26]:
# ═══ FIX: Export vào thư mục model/ của repo hiện tại ═══
print("\n💾 Đang xuất file não...")
save_dir = os.path.join(os.getcwd(), "model")
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, 'deep_learning_agent_core.keras')) 

with open(os.path.join(save_dir, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(save_dir, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

print(f"✅ HOÀN TẤT! Files đã được lưu tại: {save_dir}")


💾 Đang xuất file não...
✅ HOÀN TẤT! Files đã được lưu tại: d:\AI\ai_security\model


## 🧪 8. Chạy Thử Nghiệm Thực Tế

In [ ]:
def test_ai(payload):
    seq = tokenizer.texts_to_sequences([str(payload)])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred_probs = model.predict(pad, verbose=0)[0]
    pred_class = np.argmax(pred_probs)
    label = le.inverse_transform([pred_class])[0]
    confidence = pred_probs[pred_class] * 100
    print(f"Payload: {payload}\n=> Phát hiện: [ {label} ] (Độ tự tin: {confidence:.2f}%)\n")

print("--- 🛡️ TEST HACKER ---")
test_ai("admin' OR 1")
test_ai("<img src='x' onerror='alert(1)'>")
test_ai("test && cat /etc/passwd")
test_ai("../../../../etc/shadow")
test_ai("http://169.254.169.254/latest/meta-data/")

print("--- 🔬 TEST MODERN ATTACKS (NEW) ---")
test_ai("{{7*7}}")
test_ai('{"$gt": ""}')
test_ai("<!DOCTYPE foo [<!ENTITY xxe SYSTEM 'file:///etc/passwd'>]><foo>&xxe;</foo>")
test_ai("eyJhbGciOiJub25lIn0.eyJ1c2VyIjoiYWRtaW4ifQ.")

print("--- 🟢 TEST NORMAL ---")
test_ai("Xin chào, tôi muốn tìm tài liệu NCKH")
test_ai("https://www.google.com/search?q=cat")


--- 🛡️ TEST HACKER ---
Payload: admin' OR 1
=> Phát hiện: [ SQLi ] (Độ tự tin: 99.51%)

Payload: <img src='x' onerror='alert(1)'>
=> Phát hiện: [ XSS ] (Độ tự tin: 100.00%)

Payload: test && cat /etc/passwd
=> Phát hiện: [ Command Injection ] (Độ tự tin: 99.36%)

Payload: ../../../../etc/shadow
=> Phát hiện: [ Path Traversal ] (Độ tự tin: 50.67%)

Payload: http://169.254.169.254/latest/meta-data/
=> Phát hiện: [ SSRF ] (Độ tự tin: 100.00%)

--- 🔬 TEST MODERN ATTACKS (NEW) ---
Payload: {{7*7}}
=> Phát hiện: [ SSTI ] (Độ tự tin: 98.30%)

Payload: {"$gt": ""}
=> Phát hiện: [ NoSQLi ] (Độ tự tin: 99.94%)

Payload: <!DOCTYPE foo [<!ENTITY xxe SYSTEM 'file:///etc/passwd'>]><foo>&xxe;</foo>
=> Phát hiện: [ XXE ] (Độ tự tin: 99.96%)

Payload: eyJhbGciOiJub25lIn0.eyJ1c2VyIjoiYWRtaW4ifQ.
=> Phát hiện: [ JWTAuth ] (Độ tự tin: 100.00%)

--- 🟢 TEST NORMAL ---
Payload: Xin chào, tôi muốn tìm tài liệu NCKH
=> Phát hiện: [ Normal ] (Độ tự tin: 100.00%)

Payload: https://www.google.com/search?q=cat
=> 

: 